# PHASE 09: IMPROVED MOBILENETV2 ARCHITECTURE STUDY

Phase này cải tiến MobileNetV2 baseline từ Phase 08 theo hướng lightweight architecture cho PlantVillage Disease Classification.

Ý tưởng chính:

- MobileNetV2 pretrained backbone học visual features tổng quát.
- Residual enhancement branch học local disease textures.
- SE/CBAM attention giúp model tập trung hơn vào vùng bệnh.
- WeightedRandomSampler + Light Augmentation + EarlyStopping tiếp tục được dùng làm winner imbalance strategy.


# 01. Thiết lập môi trường và import helper dùng chung

Notebook này dùng lại `src.models.get_improved_mobilenet_v2`, `src.train.train_model`, `src.train.evaluate_model` và các data utilities đã chuẩn hóa từ các phase trước.


In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_DIR = Path('/Users/huynh/codes/kpdl/plan')
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from src.data import (
    ProjectPaths,
    ensure_directories,
    load_metadata,
    get_class_counts,
    get_num_classes,
    compute_inverse_frequency_weights,
    build_data_loader,
    build_weighted_sampler,
)
from src.models import get_mobilenet_v2, get_improved_mobilenet_v2
from src.train import train_model, evaluate_model
from src.utils import (
    set_seed,
    get_device,
    get_pin_memory,
    get_num_workers,
    get_basic_transform,
    get_light_augmentation_transform,
    dataframe_to_markdown_safe,
    write_text_report,
)
from src.visualization import set_publication_style, plot_confusion_heatmap

SEED = 42
set_seed(SEED)
set_publication_style()

DEVICE = get_device()
print(f'Thiết bị sử dụng: {DEVICE}')

PATHS = ProjectPaths(PROJECT_DIR)
BASE_DIR = PATHS.base_dir
METADATA_DIR = PATHS.metadata_dir
RESULTS_DIR = PATHS.results_dir('09_improved_mobilenetv2_architecture_study')
REPORTS_DIR = PATHS.reports_dir('09_improved_mobilenetv2_architecture_study')
PHASE09_DIR = RESULTS_DIR / 'phase_09_improved_mobilenetv2'
PHASE09_FIGURE_DIR = PHASE09_DIR / 'figures'
PHASE09_TABLE_DIR = PHASE09_DIR / 'tables'
PHASE09_REPORT_DIR = PHASE09_DIR / 'reports'
PHASE09_MODEL_DIR = PHASE09_DIR / 'saved_models'
PHASE09_GRADCAM_DIR = PHASE09_DIR / 'gradcam'

ensure_directories(
    RESULTS_DIR,
    REPORTS_DIR,
    PHASE09_DIR,
    PHASE09_FIGURE_DIR,
    PHASE09_TABLE_DIR,
    PHASE09_REPORT_DIR,
    PHASE09_MODEL_DIR,
    PHASE09_GRADCAM_DIR,
)

# Phase 09 dùng toàn bộ data như Phase 08 để đánh giá kiến trúc nghiêm túc.
IMAGE_SIZE = 128
NUM_WORKERS = get_num_workers(DEVICE, notebook_safe=True)
PIN_MEMORY = get_pin_memory(DEVICE)

FORCE_TRAIN = True
BATCH_SIZE = 64
EARLY_STOPPING_PATIENCE = 3

FEATURE_EXTRACTION_EPOCHS = 5
FINE_TUNING_EPOCHS = 10
FEATURE_EXTRACTION_LR = 1e-3
FINE_TUNING_LR = 1e-5
DROPOUT_P = 0.3
BRANCH_CHANNELS = 96
RUN_CBAM_EXPERIMENT = False



# 02. Nạp toàn bộ train/val/test data

Phase 09 dùng toàn bộ metadata train/val/test, không dùng subset. Minority classes được detect tự động từ phân phối train.


In [ ]:
train_df, val_df, test_df, class_map_df = load_metadata(METADATA_DIR)
NUM_CLASSES = get_num_classes(class_map_df=class_map_df)
class_names_list = class_map_df.sort_values('class_id')['class_name'].tolist()

train_counts = get_class_counts(train_df)
imbalance_ratio = train_counts.max() / train_counts.min()
minority_threshold = train_counts.quantile(0.25)
minority_ids = train_counts[train_counts <= minority_threshold].index.tolist()

print('[FULL DATA MODE] Phase 09 dùng toàn bộ train/val/test metadata.')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Số class: {NUM_CLASSES}')
print(f'Imbalance ratio: {imbalance_ratio:.2f}')
print(f'Minority class IDs: {minority_ids}')



# 03. Transform, WeightedRandomSampler và DataLoader

Toàn bộ kiến trúc trong Phase 09 dùng winner imbalance strategy: WeightedRandomSampler + Light Augmentation + EarlyStopping theo Validation Macro F1.


In [ ]:
train_transform = get_light_augmentation_transform(IMAGE_SIZE)
val_test_transform = get_basic_transform(IMAGE_SIZE)

sampler_weights = compute_inverse_frequency_weights(train_counts, num_classes=NUM_CLASSES)
weight_table = pd.DataFrame({
    'class_id': range(NUM_CLASSES),
    'class_name': class_names_list,
    'train_count': [int(train_counts.get(class_id, 0)) for class_id in range(NUM_CLASSES)],
    'sampler_weight': sampler_weights,
    'is_minority': [class_id in minority_ids for class_id in range(NUM_CLASSES)],
})
weight_table.to_csv(PHASE09_TABLE_DIR / 'phase09_sampler_weights.csv', index=False)
display(weight_table.sort_values('train_count').head(15))


def build_phase09_loaders():
    sampler = build_weighted_sampler(train_df, sampler_weights, replacement=True)

    train_loader = build_data_loader(
        train_df,
        BASE_DIR,
        'color',
        train_transform,
        batch_size=BATCH_SIZE,
        shuffle=False,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    val_loader = build_data_loader(
        val_df,
        BASE_DIR,
        'color',
        val_test_transform,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    test_loader = build_data_loader(
        test_df,
        BASE_DIR,
        'color',
        val_test_transform,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    return train_loader, val_loader, test_loader



# 04. Model factory và architecture variants

Ablation study gồm:

- `mobilenetv2_baseline`: MobileNetV2 gốc.
- `mobilenetv2_residual`: thêm residual enhancement branch.
- `mobilenetv2_residual_se`: thêm residual branch + SE attention.
- `mobilenetv2_residual_cbam`: optional lightweight CBAM.


In [ ]:
def build_phase09_model(model_name):
    """Khởi tạo model cho Phase 09."""
    if model_name == 'mobilenetv2_baseline':
        return get_mobilenet_v2(
            num_classes=NUM_CLASSES,
            freeze_features=True,
            dropout_p=0.2,
        )
    if model_name == 'mobilenetv2_residual':
        return get_improved_mobilenet_v2(
            num_classes=NUM_CLASSES,
            attention=None,
            branch_channels=BRANCH_CHANNELS,
            dropout_p=DROPOUT_P,
            freeze_backbone=True,
        )
    if model_name == 'mobilenetv2_residual_se':
        return get_improved_mobilenet_v2(
            num_classes=NUM_CLASSES,
            attention='se',
            branch_channels=BRANCH_CHANNELS,
            dropout_p=DROPOUT_P,
            freeze_backbone=True,
        )
    if model_name == 'mobilenetv2_residual_cbam':
        return get_improved_mobilenet_v2(
            num_classes=NUM_CLASSES,
            attention='cbam',
            branch_channels=BRANCH_CHANNELS,
            dropout_p=DROPOUT_P,
            freeze_backbone=True,
        )
    raise ValueError(f'Unknown model_name: {model_name}')


def set_backbone_trainable(model, trainable):
    """Unfreeze/freeze MobileNetV2 backbone cho cả baseline và improved model."""
    if hasattr(model, 'features'):
        for parameter in model.features.parameters():
            parameter.requires_grad = trainable
    elif hasattr(model, 'set_backbone_trainable'):
        model.set_backbone_trainable(trainable)
    else:
        raise ValueError('Model không có MobileNetV2 backbone rõ ràng.')


def count_parameters(model, trainable_only=False):
    parameters = model.parameters()
    if trainable_only:
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    return sum(parameter.numel() for parameter in parameters)



# 05. Experiment design

CBAM được triển khai nhưng mặc định `RUN_CBAM_EXPERIMENT=False` để tiết kiệm tài nguyên. Có thể bật lên nếu cần ablation đầy đủ hơn.


In [ ]:
ARCHITECTURE_EXPERIMENTS = [
    {
        'name': 'mobilenetv2_baseline',
        'description': 'Original MobileNetV2 pretrained backbone',
    },
    {
        'name': 'mobilenetv2_residual',
        'description': 'MobileNetV2 + lightweight residual enhancement branch',
    },
    {
        'name': 'mobilenetv2_residual_se',
        'description': 'MobileNetV2 + residual branch + SE attention',
    },
]

if RUN_CBAM_EXPERIMENT:
    ARCHITECTURE_EXPERIMENTS.append({
        'name': 'mobilenetv2_residual_cbam',
        'description': 'MobileNetV2 + residual branch + lightweight CBAM attention',
    })

experiment_design_df = pd.DataFrame(ARCHITECTURE_EXPERIMENTS)
experiment_design_df.to_csv(PHASE09_TABLE_DIR / 'phase09_architecture_experiment_design.csv', index=False)
display(experiment_design_df)



# 06. Two-stage training pipeline

Stage 1 freeze backbone để train classifier/fusion layers. Stage 2 unfreeze backbone và fine-tune với learning rate nhỏ.


In [ ]:
architecture_results = {}
architecture_histories = {}


def merge_stage_histories(stage1_history, stage2_history):
    merged = {}
    for key in ['train_loss', 'val_loss', 'val_f1', 'val_acc']:
        merged[key] = stage1_history.get(key, []) + stage2_history.get(key, [])
    merged['stage'] = ['feature_extraction'] * len(stage1_history.get('train_loss', [])) + ['fine_tuning'] * len(stage2_history.get('train_loss', []))
    return merged


def save_history(history, experiment_name):
    history_df = pd.DataFrame({
        'epoch': range(1, len(history['train_loss']) + 1),
        'train_loss': history['train_loss'],
        'val_loss': history['val_loss'],
        'val_f1': history['val_f1'],
        'val_acc': history['val_acc'],
        'stage': history.get('stage', ['single_stage'] * len(history['train_loss'])),
    })
    history_df.to_csv(PHASE09_TABLE_DIR / f'{experiment_name}_training_history.csv', index=False)


def train_two_stage_model(model, experiment_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    final_path = PHASE09_MODEL_DIR / f'{experiment_name}_{IMAGE_SIZE}px_best.pth'
    stage1_path = PHASE09_MODEL_DIR / f'{experiment_name}_{IMAGE_SIZE}px_stage1.pth'

    set_backbone_trainable(model, trainable=False)
    stage1_optimizer = optim.Adam(filter(lambda parameter: parameter.requires_grad, model.parameters()), lr=FEATURE_EXTRACTION_LR)
    print(f'Stage 1 trainable params: {count_parameters(model, trainable_only=True):,}')
    stage1_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=stage1_optimizer,
        epochs=FEATURE_EXTRACTION_EPOCHS,
        device=DEVICE,
        save_path=stage1_path,
        patience=EARLY_STOPPING_PATIENCE,
        use_early_stopping=True,
    )
    model.load_state_dict(torch.load(stage1_path, map_location=DEVICE))

    set_backbone_trainable(model, trainable=True)
    stage2_optimizer = optim.Adam(model.parameters(), lr=FINE_TUNING_LR)
    print(f'Stage 2 trainable params: {count_parameters(model, trainable_only=True):,}')
    stage2_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=stage2_optimizer,
        epochs=FINE_TUNING_EPOCHS,
        device=DEVICE,
        save_path=final_path,
        patience=EARLY_STOPPING_PATIENCE,
        use_early_stopping=True,
    )
    model.load_state_dict(torch.load(final_path, map_location=DEVICE))
    return model, merge_stage_histories(stage1_history, stage2_history)



# 07. Run architecture experiment

Mỗi model train độc lập, lưu checkpoint, predictions, confusion matrix, per-class metrics và parameter count.


In [ ]:
def save_predictions_and_metrics(result, experiment_name):
    pd.DataFrame({
        'true_label': result['labels'],
        'predicted_label': result['preds'],
    }).to_csv(PHASE09_TABLE_DIR / f'{experiment_name}_predictions.csv', index=False)

    per_class_df = pd.DataFrame({
        'Experiment': experiment_name,
        'Class_ID': range(NUM_CLASSES),
        'Class': class_names_list,
        'Train Sample Count': [int(train_counts.get(class_id, 0)) for class_id in range(NUM_CLASSES)],
        'Is Minority': [class_id in minority_ids for class_id in range(NUM_CLASSES)],
        'Precision': result['per_class_precision'],
        'Recall': result['per_class_recall'],
        'F1-score': result['per_class_f1'],
    })
    per_class_df.to_csv(PHASE09_TABLE_DIR / f'{experiment_name}_per_class_metrics.csv', index=False)


def run_architecture_experiment(config):
    experiment_name = config['name']
    set_seed(SEED)
    train_loader, val_loader, test_loader = build_phase09_loaders()
    model = build_phase09_model(experiment_name).to(DEVICE)

    print('\n' + '=' * 80)
    print(f'ARCHITECTURE EXPERIMENT: {experiment_name}')
    print(config['description'])
    print('=' * 80)
    print(f'Total parameters: {count_parameters(model):,}')

    final_path = PHASE09_MODEL_DIR / f'{experiment_name}_{IMAGE_SIZE}px_best.pth'
    if (not FORCE_TRAIN) and final_path.exists():
        set_backbone_trainable(model, trainable=True)
        model.load_state_dict(torch.load(final_path, map_location=DEVICE))
        history = None
        print(f'Đã nạp checkpoint: {final_path}')
    else:
        model, history = train_two_stage_model(model, experiment_name, train_loader, val_loader)
        architecture_histories[experiment_name] = history
        save_history(history, experiment_name)

    result = evaluate_model(model, test_loader, DEVICE, num_classes=NUM_CLASSES)
    architecture_results[experiment_name] = result
    save_predictions_and_metrics(result, experiment_name)

    plot_confusion_heatmap(
        result['labels'],
        result['preds'],
        labels=list(range(NUM_CLASSES)),
        normalize='true',
        title=f'Normalized Confusion Matrix - {experiment_name}',
        save_path=PHASE09_FIGURE_DIR / f'{experiment_name}_confusion_matrix_normalized.png',
        figsize=(18, 15),
    )

    print(f"[RESULT {experiment_name}] Acc={result['accuracy']:.4f} | Macro F1={result['macro_f1']:.4f}")
    return result



# 08. Chạy ablation experiments

Cell này có thể tốn thời gian vì mỗi kiến trúc dùng MobileNetV2 pretrained và two-stage training.


In [ ]:
for experiment_config in ARCHITECTURE_EXPERIMENTS:
    run_architecture_experiment(experiment_config)



# 09. Ablation study table

Bảng ablation cho biết residual branch và attention có đóng góp thật sự không, đồng thời theo dõi số parameters.


In [ ]:
def summarize_architecture_results(results):
    records = []
    for experiment_name, result in results.items():
        model = build_phase09_model(experiment_name)
        minority_recall = np.mean([result['per_class_recall'][class_id] for class_id in minority_ids])
        worst_recall = np.min(result['per_class_recall'])
        records.append({
            'Model': experiment_name,
            'Params': count_parameters(model),
            'Accuracy': result['accuracy'],
            'Macro F1': result['macro_f1'],
            'Macro Precision': result['precision'],
            'Macro Recall': result['recall'],
            'Minority Recall': minority_recall,
            'Worst Recall': worst_recall,
        })
    return pd.DataFrame(records).sort_values('Macro F1', ascending=False).reset_index(drop=True)


ablation_df = summarize_architecture_results(architecture_results)
ablation_df.to_csv(PHASE09_TABLE_DIR / 'improved_mobilenetv2_ablation_results.csv', index=False)
display(ablation_df)

best_model_name = ablation_df.sort_values(['Macro F1', 'Minority Recall'], ascending=False).iloc[0]['Model']
print(f'Best architecture: {best_model_name}')



# 10. Model complexity analysis

Phân tích parameter count, estimated parameter memory và inference time trung bình.


In [ ]:
def measure_inference_time(model, image_size=IMAGE_SIZE, warmup_runs=5, timed_runs=20):
    model = model.to(DEVICE)
    model.eval()
    dummy_input = torch.randn(1, 3, image_size, image_size).to(DEVICE)

    with torch.no_grad():
        for _ in range(warmup_runs):
            _ = model(dummy_input)

        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        for _ in range(timed_runs):
            _ = model(dummy_input)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_time = time.perf_counter() - start_time

    return (elapsed_time / timed_runs) * 1000


complexity_records = []
for config in ARCHITECTURE_EXPERIMENTS:
    model = build_phase09_model(config['name'])
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    estimated_memory_mb = total_params * 4 / (1024 ** 2)
    inference_time_ms = measure_inference_time(model)
    complexity_records.append({
        'Model': config['name'],
        'Total Params': total_params,
        'Trainable Params Stage 1': trainable_params,
        'Estimated Param Memory MB': estimated_memory_mb,
        'Inference Time ms/image': inference_time_ms,
    })

complexity_df = pd.DataFrame(complexity_records)
complexity_df.to_csv(PHASE09_TABLE_DIR / 'improved_mobilenetv2_complexity.csv', index=False)
display(complexity_df)



# 11. Visualization: ablation metrics và complexity

Các biểu đồ này dùng cho report/paper để so sánh performance và efficiency.


In [ ]:
metric_columns = ['Accuracy', 'Macro F1', 'Minority Recall', 'Worst Recall']
plot_df = ablation_df.melt(id_vars='Model', value_vars=metric_columns, var_name='Metric', value_name='Score')

plt.figure(figsize=(13, 6))
ax = sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric')
ax.set_title('Improved-MobileNetV2 Ablation Metrics', fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(PHASE09_FIGURE_DIR / 'improved_mobilenetv2_ablation_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=complexity_df, x='Model', y='Total Params', color='#4C78A8')
ax.set_title('Parameter Count Comparison', fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Total Parameters')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(PHASE09_FIGURE_DIR / 'improved_mobilenetv2_parameter_count.png', dpi=300, bbox_inches='tight')
plt.show()



# 12. Learning curve analysis

Learning curves giúp kiểm tra residual/attention có làm training ổn định hơn hay gây overfitting không.


In [ ]:
if len(architecture_histories) == 0:
    print('Không có training history. Nếu load checkpoint bằng FORCE_TRAIN=False thì curves sẽ không có.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for experiment_name, history in architecture_histories.items():
        epochs_range = range(1, len(history['train_loss']) + 1)
        axes[0].plot(epochs_range, history['train_loss'], marker='o', label=experiment_name)
        axes[1].plot(epochs_range, history['val_loss'], marker='o', label=experiment_name)
        axes[2].plot(epochs_range, history['val_f1'], marker='o', label=experiment_name)

    axes[0].set_title('Train Loss')
    axes[1].set_title('Validation Loss')
    axes[2].set_title('Validation Macro F1')
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(True, linestyle='--', alpha=0.4)
    axes[0].set_ylabel('Loss')
    axes[1].set_ylabel('Loss')
    axes[2].set_ylabel('Macro F1')
    axes[2].set_ylim(0, 1)
    axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    fig.suptitle('Improved-MobileNetV2 Learning Curves', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PHASE09_FIGURE_DIR / 'improved_mobilenetv2_learning_curves.png', dpi=300, bbox_inches='tight')
    plt.show()



# 13. Per-class analysis và error analysis

Phân tích Recall/F1 theo class để xác định class hưởng lợi mạnh nhất và class vẫn khó học.


In [ ]:
per_class_records = []
for experiment_name, result in architecture_results.items():
    for class_id in range(NUM_CLASSES):
        per_class_records.append({
            'Model': experiment_name,
            'Class_ID': class_id,
            'Class': class_names_list[class_id],
            'Train Sample Count': int(train_counts.get(class_id, 0)),
            'Is Minority': class_id in minority_ids,
            'Recall': result['per_class_recall'][class_id],
            'F1-score': result['per_class_f1'][class_id],
        })

per_class_architecture_df = pd.DataFrame(per_class_records)
per_class_architecture_df.to_csv(PHASE09_TABLE_DIR / 'improved_mobilenetv2_per_class_metrics_all.csv', index=False)

recall_heatmap_df = per_class_architecture_df.pivot(index='Class', columns='Model', values='Recall')
plt.figure(figsize=(10, max(10, NUM_CLASSES * 0.32)))
sns.heatmap(recall_heatmap_df, cmap='YlGnBu', vmin=0, vmax=1, cbar_kws={'label': 'Recall'})
plt.title('Per-class Recall Heatmap - Improved-MobileNetV2 Ablation', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Class')
plt.tight_layout()
plt.savefig(PHASE09_FIGURE_DIR / 'improved_mobilenetv2_per_class_recall_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

if 'mobilenetv2_baseline' in architecture_results and best_model_name in architecture_results:
    baseline_recall = architecture_results['mobilenetv2_baseline']['per_class_recall']
    best_recall = architecture_results[best_model_name]['per_class_recall']
    improvement_df = pd.DataFrame({
        'Class_ID': range(NUM_CLASSES),
        'Class': class_names_list,
        'Train Sample Count': [int(train_counts.get(class_id, 0)) for class_id in range(NUM_CLASSES)],
        'Is Minority': [class_id in minority_ids for class_id in range(NUM_CLASSES)],
        'Baseline Recall': baseline_recall,
        f'{best_model_name} Recall': best_recall,
        'Recall Improvement': best_recall - baseline_recall,
    }).sort_values('Recall Improvement', ascending=False)
    improvement_df.to_csv(PHASE09_TABLE_DIR / 'improved_mobilenetv2_recall_improvement.csv', index=False)

    print('Top class cải thiện mạnh nhất:')
    display(improvement_df.head(15))

    print('Hardest classes dưới best improved model:')
    hardest_df = improvement_df.sort_values(f'{best_model_name} Recall').head(15)
    display(hardest_df)

misclassification_records = []
for experiment_name, result in architecture_results.items():
    labels = np.array(result['labels'])
    preds = np.array(result['preds'])
    wrong_mask = labels != preds
    for class_id in range(NUM_CLASSES):
        class_wrong = np.sum((labels == class_id) & wrong_mask)
        class_total = np.sum(labels == class_id)
        misclassification_records.append({
            'Model': experiment_name,
            'Class_ID': class_id,
            'Class': class_names_list[class_id],
            'Misclassified Count': class_wrong,
            'Class Total': class_total,
            'Misclassification Rate': class_wrong / class_total if class_total > 0 else 0,
        })

misclassification_df = pd.DataFrame(misclassification_records)
misclassification_df.to_csv(PHASE09_TABLE_DIR / 'improved_mobilenetv2_misclassification_histogram.csv', index=False)



# 14. Grad-CAM và Grad-CAM++ utilities

Grad-CAM giúp kiểm tra model có tập trung vào vùng bệnh trên lá hay không. Grad-CAM++ được triển khai như biến thể nhấn mạnh localization hơn với hard examples.


In [ ]:
class GradCAMBase:
    """Base class for Grad-CAM style visualization."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def _normalize_cam(self, cam):
        cam = F.relu(cam)
        cam_min = cam.min()
        cam_max = cam.max()
        return (cam - cam_min) / (cam_max - cam_min + 1e-8)


class GradCAM(GradCAMBase):
    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad()
        logits = self.model(input_tensor)
        if target_class is None:
            target_class = int(logits.argmax(dim=1).item())
        score = logits[:, target_class].sum()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        return self._normalize_cam(cam)[0, 0].cpu().numpy(), target_class


class GradCAMPlusPlus(GradCAMBase):
    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad()
        logits = self.model(input_tensor)
        if target_class is None:
            target_class = int(logits.argmax(dim=1).item())
        score = logits[:, target_class].sum()
        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations
        gradients_power_2 = gradients ** 2
        gradients_power_3 = gradients_power_2 * gradients
        denominator = 2 * gradients_power_2 + (activations * gradients_power_3).sum(dim=(2, 3), keepdim=True)
        alpha = gradients_power_2 / (denominator + 1e-8)
        positive_gradients = F.relu(gradients)
        weights = (alpha * positive_gradients).sum(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        return self._normalize_cam(cam)[0, 0].cpu().numpy(), target_class


def get_gradcam_target_layer(model):
    """Chọn layer cuối phù hợp cho baseline hoặc improved MobileNetV2."""
    if hasattr(model, 'backbone_features'):
        return model.backbone_features[-1]
    if hasattr(model, 'features'):
        return model.features[-1]
    raise ValueError('Không tìm thấy target layer cho Grad-CAM.')


def load_image_for_cam(row):
    image_path = BASE_DIR / 'color' / row['relative_path']
    image = Image.open(image_path).convert('RGB')
    input_tensor = val_test_transform(image).unsqueeze(0).to(DEVICE)
    resized_image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    image_array = np.array(resized_image) / 255.0
    return input_tensor, image_array


def overlay_cam(image_array, cam, alpha=0.45):
    heatmap = plt.get_cmap('jet')(cam)[..., :3]
    overlay = (1 - alpha) * image_array + alpha * heatmap
    return np.clip(overlay, 0, 1)



# 15. Grad-CAM visualization cho baseline và improved model

Cell này chọn một số mẫu misclassified/hard examples từ test set và so sánh heatmap giữa MobileNetV2 baseline và best improved model.


In [ ]:
def load_trained_model_for_cam(model_name):
    model = build_phase09_model(model_name).to(DEVICE)
    set_backbone_trainable(model, trainable=True)
    checkpoint_path = PHASE09_MODEL_DIR / f'{model_name}_{IMAGE_SIZE}px_best.pth'
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    return model


def select_hard_examples(result, max_examples=4):
    labels = np.array(result['labels'])
    preds = np.array(result['preds'])
    wrong_indices = np.where(labels != preds)[0]
    if len(wrong_indices) == 0:
        wrong_indices = np.arange(min(max_examples, len(test_df)))
    return wrong_indices[:max_examples]


def visualize_gradcam_comparison(baseline_name='mobilenetv2_baseline', improved_name=None, max_examples=4):
    if improved_name is None:
        improved_name = best_model_name

    if baseline_name not in architecture_results or improved_name not in architecture_results:
        print('Cần chạy baseline và improved model trước khi vẽ Grad-CAM.')
        return

    baseline_model = load_trained_model_for_cam(baseline_name)
    improved_model = load_trained_model_for_cam(improved_name)

    baseline_gradcam = GradCAM(baseline_model, get_gradcam_target_layer(baseline_model))
    improved_gradcam = GradCAM(improved_model, get_gradcam_target_layer(improved_model))
    improved_gradcam_pp = GradCAMPlusPlus(improved_model, get_gradcam_target_layer(improved_model))

    selected_indices = select_hard_examples(architecture_results[improved_name], max_examples=max_examples)

    for rank, sample_index in enumerate(selected_indices, start=1):
        row = test_df.iloc[int(sample_index)]
        input_tensor, image_array = load_image_for_cam(row)

        baseline_cam, baseline_pred = baseline_gradcam(input_tensor.clone())
        improved_cam, improved_pred = improved_gradcam(input_tensor.clone())
        improved_cam_pp, improved_pred_pp = improved_gradcam_pp(input_tensor.clone())

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(image_array)
        axes[0].set_title(f'Original\nTrue: {row["class_id"]}')
        axes[1].imshow(overlay_cam(image_array, baseline_cam))
        axes[1].set_title(f'Baseline Grad-CAM\nPred: {baseline_pred}')
        axes[2].imshow(overlay_cam(image_array, improved_cam))
        axes[2].set_title(f'Improved Grad-CAM\nPred: {improved_pred}')
        axes[3].imshow(overlay_cam(image_array, improved_cam_pp))
        axes[3].set_title(f'Improved Grad-CAM++\nPred: {improved_pred_pp}')

        for ax in axes:
            ax.axis('off')

        plt.tight_layout()
        save_path = PHASE09_GRADCAM_DIR / f'gradcam_comparison_{rank}_{improved_name}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

    baseline_gradcam.remove_hooks()
    improved_gradcam.remove_hooks()
    improved_gradcam_pp.remove_hooks()


visualize_gradcam_comparison(max_examples=4)



# 16. Discussion học thuật

Phần này diễn giải tác động của residual branch, attention, transfer learning, imbalance handling và deployment efficiency.


In [ ]:
best_row = ablation_df[ablation_df['Model'] == best_model_name].iloc[0]
baseline_row = ablation_df[ablation_df['Model'] == 'mobilenetv2_baseline'].iloc[0]
macro_f1_gain = best_row['Macro F1'] - baseline_row['Macro F1']
minority_recall_gain = best_row['Minority Recall'] - baseline_row['Minority Recall']
param_gain = best_row['Params'] - baseline_row['Params']

if macro_f1_gain > 0 and minority_recall_gain > 0:
    architecture_statement = 'Improved architecture cho thấy hiệu quả bổ trợ: tăng đồng thời Macro F1 và Minority Recall so với MobileNetV2 baseline.'
else:
    architecture_statement = 'Kết quả chưa cho thấy cải thiện đồng thời rõ ràng; MobileNetV2 pretrained baseline có thể đã đủ mạnh hoặc residual/attention branch cần tinh chỉnh thêm.'

discussion_text = f"""
THẢO LUẬN HỌC THUẬT
===================

Phase 09 đề xuất Improved-MobileNetV2 bằng cách kết hợp MobileNetV2 pretrained backbone với residual enhancement branch và attention nhẹ. MobileNetV2 cung cấp feature extractor mạnh, học được các visual primitives từ ImageNet. Residual branch được thiết kế nông và nhẹ để học thêm local disease patterns như đốm bệnh, texture tổn thương và biến đổi màu sắc trên lá.

SE attention hoặc CBAM nhẹ có mục tiêu giúp model tăng trọng số cho các kênh hoặc vùng không gian có thông tin phân biệt bệnh. Điều này đặc biệt quan trọng trong plant disease classification vì vùng bệnh có thể nhỏ, không nằm ở trung tâm ảnh hoặc bị nhiễu bởi nền lá khỏe.

Best architecture là `{best_model_name}` với Macro F1 = {best_row['Macro F1']:.4f}, Minority Recall = {best_row['Minority Recall']:.4f}, Worst Recall = {best_row['Worst Recall']:.4f}. So với MobileNetV2 baseline, Macro F1 thay đổi {macro_f1_gain:+.4f}, Minority Recall thay đổi {minority_recall_gain:+.4f}, và số parameters thay đổi {param_gain:+,}.

{architecture_statement}

Về deployment, kiến trúc cải tiến vẫn giữ MobileNetV2 làm backbone chính và residual branch tương đối nhỏ. Vì vậy mô hình vẫn phù hợp hơn cho edge/mobile deployment so với các backbone lớn. Tuy nhiên, nếu số parameters hoặc inference time tăng mà Macro F1 không cải thiện tương ứng, kiến trúc baseline vẫn là lựa chọn hiệu quả hơn.
""".strip()

write_text_report(PHASE09_REPORT_DIR / 'improved_mobilenetv2_discussion.txt', [discussion_text])
print(discussion_text)



# 17. Xuất final report

Báo cáo gồm architecture design, methodology, ablation study, quantitative results, explainability analysis, efficiency analysis, discussion và conclusions.


In [ ]:
report_sections = []
report_sections.append('# Báo cáo Phase 09: Improved MobileNetV2 Architecture Study')
report_sections.append(f'Thời điểm tạo báo cáo: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
report_sections.append('')
report_sections.append('## 1. Architecture design')
report_sections.append('- Path 1: MobileNetV2 pretrained backbone làm feature extractor chính.')
report_sections.append('- Path 2: residual enhancement branch học local disease textures.')
report_sections.append('- Attention: SE block hoặc lightweight CBAM để tăng khả năng focus vùng bệnh.')
report_sections.append('- Fusion: concatenate backbone features và residual features, sau đó qua classifier head.')
report_sections.append('')
report_sections.append('## 2. Methodology')
report_sections.append('Tất cả model dùng WeightedRandomSampler, Light Augmentation và EarlyStopping theo Validation Macro F1. Training theo two-stage transfer learning: feature extraction rồi fine-tuning.')
report_sections.append('')
report_sections.append('## 3. Ablation study')
report_sections.append(dataframe_to_markdown_safe(ablation_df))
report_sections.append('')
report_sections.append('## 4. Model complexity')
report_sections.append(dataframe_to_markdown_safe(complexity_df))
report_sections.append('')
report_sections.append('## 5. Per-class and error analysis')
report_sections.append('Per-class Recall/F1, hardest-class ranking và misclassification histogram được lưu trong thư mục `tables/`.')
report_sections.append('')
report_sections.append('## 6. Explainability analysis')
report_sections.append('Grad-CAM và Grad-CAM++ visualizations được lưu trong thư mục `gradcam/`. Các heatmap dùng để kiểm tra model có tập trung vào vùng bệnh trên lá hay không.')
report_sections.append('')
report_sections.append('## 7. Discussion')
report_sections.append(discussion_text)
report_sections.append('')
report_sections.append('## 8. Conclusions')
report_sections.append('- MobileNetV2 pretrained là backbone mạnh cho PlantVillage classification.')
report_sections.append('- Residual enhancement branch có mục tiêu bổ sung local disease texture learning.')
report_sections.append('- SE/CBAM attention giúp kiểm tra vai trò focus vào disease-discriminative regions.')
report_sections.append('- WeightedRandomSampler vẫn quan trọng để cải thiện minority exposure.')
report_sections.append('- Kiến trúc cải tiến chỉ nên được chọn nếu cải thiện Macro F1/Minority Recall tương xứng với chi phí parameters và inference time.')

write_text_report(PHASE09_REPORT_DIR / 'improved_mobilenetv2_report.md', report_sections)
write_text_report(PHASE09_REPORT_DIR / 'improved_mobilenetv2_report.txt', report_sections)

print('\n'.join(report_sections))
print(f'Đã lưu report tại: {PHASE09_REPORT_DIR}')

